In [1]:
import os

### Write ```Dockerfile```

In [2]:
%%writefile Dockerfile

FROM python:3.9
    
# update package manager
RUN apt-get update

# update pip
RUN pip install --upgrade pip

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install -r requirements.txt

# copy script into container
COPY script.py .

# run script when image is run
ENTRYPOINT ["python3", "script.py"]

Writing Dockerfile


### Write ```requirements.txt```

In [3]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

pandas==1.2.4
boto3==1.24.59
shap==0.40.0
matplotlib==3.6.1
catboost==1.0.4
seaborn==0.11.2

Writing requirements.txt


### Write ```script.py``` to local drive

In [4]:
%%writefile script.py

import os
import sklearn.metrics as skm
import pandas as pd
import numpy as np
import boto3
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import json

# eval metrics
def generate_eval_metrics(y_true, y_hat, str_filename, str_dirname_output):
    # if doing classification
    y_hat_proba = y_hat.copy()
    flt_threshold = np.mean(y_true)
    y_hat = np.where(y_hat_proba >= flt_threshold, 1, 0)
    # dictionary of binary eval metrics
    dict_eval_metrics = {
        'accuracy': skm.accuracy_score(y_true=y_true, y_pred=y_hat),
        'precision': skm.precision_score(y_true=y_true, y_pred=y_hat),
        'recall': skm.recall_score(y_true=y_true, y_pred=y_hat),
        'f1': skm.f1_score(y_true=y_true, y_pred=y_hat),
        'roc_auc': skm.roc_auc_score(y_true=y_true, y_score=y_hat_proba),
        'pr_auc': skm.average_precision_score(y_true=y_true, y_score=y_hat_proba),
        'log_loss': skm.log_loss(y_true=y_true, y_pred=y_hat_proba),
    }
    # write to .json
    json.dump(dict_eval_metrics, open(f'{str_dirname_output}/{str_filename}', 'w'))
    # return object
    return dict_eval_metrics

# roc-auc curve
def get_roc_auc_curve(y_true, y_hat, str_dirname_output, str_filename):
    flt_roc_auc = skm.roc_auc_score(y_true=y_true, y_score=y_hat)
    # get false positive rate, true positive rate
    fpr, tpr, thresholds = skm.roc_curve(y_true=y_true, y_score=y_hat)
    # set up subplots
    fig, ax = plt.subplots(figsize=(9,7))
    # set title
    ax.set_title(f"ROC Plot - (AUC: {flt_roc_auc:0.4})")
    # set x axis label
    ax.set_xlabel('False Positive Rate')
    # set y axis label
    ax.set_ylabel('True Positive Rate')
    # set x lim
    ax.set_xlim([0,1])
    # set y lim
    ax.set_ylim([0,1])
    # create curve
    ax.plot(fpr, tpr, label='Model')
    # plot diagonal red, dotted line
    ax.plot([0,1], [0,1], color='red', linestyle=':', label='Chance')
    # create legend
    ax.legend(loc='lower right')
    # save fig
    plt.savefig(f'{str_dirname_output}/{str_filename}', bbox_inches='tight')
    # close
    plt.close()

# plot distributions of predictions
def plot_distribution_predictions(y_hat_train, y_hat_valid, str_dirname_output, str_filename):
    fig, ax = plt.subplots(figsize=(11,6))
    ax.set_title('Distribution of Predictions in Training and Validation Data')
    ax.set_xlabel('Predicted Probability')
    # train
    sns.kdeplot(y_hat_train, ax=ax, label='Train')
    # valid
    sns.kdeplot(y_hat_valid, ax=ax, label='Valid')
    plt.legend()
    plt.savefig(f'{str_dirname_output}/{str_filename}', bbox_inches='tight')
    plt.close()

# get roc auc score (helper)
def get_roc_auc(y_true, y_score):
    flt_metric = skm.roc_auc_score(
        y_true=y_true, 
        y_score=y_score,
    )
    return flt_metric

# get pr auc score (helper)
def get_pr_auc(y_true, y_score):
    flt_metric = skm.average_precision_score(
        y_true=y_true, 
        y_score=y_score,
    )
    return flt_metric

# get logloss (helper)
def get_log_loss(y_true, y_score):
    flt_metric = skm.log_loss(
        y_true=y_true, 
        y_pred=y_score,
    )
    return flt_metric

# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # download file
    boto3.client('s3').download_file(str_project, str_bucket_path, str_local_path)
    
# upload to s3
def upload_to_s3(str_local_path, str_bucket_path, str_project):
    boto3.resource('s3').Bucket(str_project).Object(str_bucket_path).upload_file(str_local_path)

# constants
str_project = '20231010-gen-xii'
str_target = 'target'
str_dirname_output = './output'
str_model = '01_ad'

# make output dir
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

###############################################################################
# HYPERPARAMETERS
###############################################################################
# get df_hyperparameters
str_filename = 'df_hyperparameters.csv'
str_uri = f's3://{str_project}/{str_model}/02_model/02_model/12_step_function/{str_filename}'
df = pd.read_csv(str_uri)
# convert to dict
dict_hyperparameters = dict(zip(df['keys'], df['values']))

# get filename for training
str_filename_train = dict_hyperparameters['STR_FILENAME_TRAIN']
print(f'Training filename: {str_filename_train}')

# get filename for valid
str_filename_valid = dict_hyperparameters['STR_FILENAME_VALID']
print(f'Valid filename: {str_filename_valid}')

# get eval metric
str_eval_metric = dict_hyperparameters['STR_EVAL_METRIC']
print(f'Eval metric: {str_eval_metric}')

##################################################################################

# get model
str_filename = 'final_model.pkl'
str_local_path = f'{str_dirname_output}/{str_filename}'
str_bucket_path = f'{str_model}/02_model/03_final_model/{str_filename}'
download_from_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)
dict_pipeline = pickle.load(open(str_local_path, 'rb'))
cls_model_inference = dict_pipeline['model_inference']
list_cols_model = list(cls_model_inference.feature_names_)
list_cols_import = list_cols_model + [str_target]

# get y_hat_train
print('Getting y_hat_train...')
str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/02_make_dfs/{str_filename_train}'
df = pd.read_parquet(str_uri, columns=list_cols_import)
y_hat_train = cls_model_inference.predict_proba(df[list_cols_model])[:,1]
y_true_train = df['target']

# get y_hat_valid
print('Getting y_hat_valid...')
str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/02_make_dfs/{str_filename_valid}'
df = pd.read_parquet(str_uri, columns=list_cols_import)
y_hat_valid = cls_model_inference.predict_proba(df[list_cols_model])[:,1]
y_true_valid = df['target']

# save memory
del df

##########
##########

# read training data
print('Reading training data...')
list_cols = [
    str_target,
]
str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/02_make_dfs/{str_filename_train}'
df = pd.read_parquet(str_uri, columns=list_cols)
df['y_hat'] = y_hat_train

# get training eval metrics
print('Getting training eval metrics...')
str_filename = 'dict_eval_metrics_train.json'
dict_eval_metrics = generate_eval_metrics(
    y_true=y_true_train, 
    y_hat=y_hat_train, 
    str_filename=str_filename, 
    str_dirname_output=str_dirname_output,
)
str_local_path = f'{str_dirname_output}/{str_filename}'
str_bucket_path = f'{str_model}/02_model/02_model/09_batch_model_eval_valid/{str_filename}'
upload_to_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)

# roc-auc curve
print('Generating training ROC-AUC curve...')
str_filename = 'plt_roc_auc_train.png'
get_roc_auc_curve(
    y_true=y_true_train, 
    y_hat=y_hat_train,
    str_filename=str_filename,
    str_dirname_output=str_dirname_output,
)
str_local_path = f'{str_dirname_output}/{str_filename}'
str_bucket_path = f'{str_model}/02_model/02_model/09_batch_model_eval_valid/{str_filename}'
upload_to_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)

# lift chart
df_eval = pd.DataFrame({
    'y_hat': y_hat_train,
    'target': y_true_train,
})
# quantile
df_eval['quantile_rank'] = pd.qcut(df_eval['y_hat'], 20, labels=False, duplicates='drop')
# group
df_eval = df_eval.groupby('quantile_rank', as_index=False).agg({
    'target': 'mean',
})
# plot
fig, ax = plt.subplots(figsize=(9,5))
ax.set_title('Mean Target (actual) by Prediction Quantile Rank for Train')
ax.set_xlabel('Quantile Rank')
ax.set_ylabel('Mean Target')
ax.plot(df_eval['quantile_rank'].astype(str), df_eval[str_target])
# save
str_filename = 'plt_lift_train.png'
str_local_path = f'{str_dirname_output}/{str_filename}'
plt.savefig(str_local_path, bbox_inches='tight')
# upload
str_bucket_path = f'{str_model}/02_model/02_model/09_batch_model_eval_valid/{str_filename}'
upload_to_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)

# save memory
del df

##########
##########

# read validation data
print('Reading validation data...')
list_cols = [
    str_target,
]
str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/02_make_dfs/{str_filename_valid}'
df = pd.read_parquet(str_uri, columns=list_cols)
df['y_hat'] = y_hat_valid

# get validation eval metrics
print('Getting validation eval metrics...')
str_filename = 'dict_eval_metrics_valid.json'
dict_eval_metrics = generate_eval_metrics(
    y_true=y_true_valid, 
    y_hat=y_hat_valid, 
    str_filename=str_filename, 
    str_dirname_output=str_dirname_output,
)
str_local_path = f'{str_dirname_output}/{str_filename}'
str_bucket_path = f'{str_model}/02_model/02_model/09_batch_model_eval_valid/{str_filename}'
upload_to_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)

# roc-auc curve
print('Generating validation ROC-AUC curve...')
str_filename = 'plt_roc_auc_valid.png'
get_roc_auc_curve(
    y_true=y_true_valid, 
    y_hat=y_hat_valid,
    str_filename=str_filename,
    str_dirname_output=str_dirname_output,
)
str_local_path = f'{str_dirname_output}/{str_filename}'
str_bucket_path = f'{str_model}/02_model/02_model/09_batch_model_eval_valid/{str_filename}'
upload_to_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)

# lift chart
df_eval = pd.DataFrame({
    'y_hat': y_hat_valid,
    'target': y_true_valid,
})
# quantile
df_eval['quantile_rank'] = pd.qcut(df_eval['y_hat'], 20, labels=False, duplicates='drop')
# group
df_eval = df_eval.groupby('quantile_rank', as_index=False).agg({
    'target': 'mean',
})
# plot
fig, ax = plt.subplots(figsize=(9,5))
ax.set_title('Mean Target (actual) by Prediction Quantile Rank for Valid')
ax.set_xlabel('Quantile Rank')
ax.set_ylabel('Mean Target')
ax.plot(df_eval['quantile_rank'].astype(str), df_eval[str_target])
# save
str_filename = 'plt_lift_valid.png'
str_local_path = f'{str_dirname_output}/{str_filename}'
plt.savefig(str_local_path, bbox_inches='tight')
# upload
str_bucket_path = f'{str_model}/02_model/02_model/09_batch_model_eval_valid/{str_filename}'
upload_to_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)

##########
##########

# distribution of predictions
print('Plottiong distribution of predictions...')
str_filename = 'plt_dist_yhat.png'
plot_distribution_predictions(
    y_hat_train=y_hat_train, 
    y_hat_valid=y_hat_valid, 
    str_dirname_output=str_dirname_output, 
    str_filename=str_filename,
)
str_local_path = f'{str_dirname_output}/{str_filename}'
str_bucket_path = f'{str_model}/02_model/02_model/09_batch_model_eval_valid/{str_filename}'
upload_to_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)

Writing script.py


### Build and push to ECR

In [5]:
%%sh

# image name
image=genxii-ad-eval-valid

# Get the account number associated with the current IAM credentials
account=$(aws sts get-caller-identity --query Account --output text)

# did we have an error?
if [ $? -ne 0 ]
then
    exit 255
fi

# Get the region defined in the current configuration (default to us-west-2 if none defined)
region=$(aws configure get region)
region=${region:-us-west-2}

# get destination of repo
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# If the repository doesn't exist in ECR, create it.
aws ecr describe-repositories --repository-names "${image}" > /dev/null 2>&1

# if it doesnt exist...create it
if [ $? -ne 0 ]
then
    aws ecr create-repository --repository-name "${image}" > /dev/null
fi

# Get the login command from ECR and execute it directly
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# Build the docker image locally with the image name and then push it to ECR
# with the full name.

# build and add tag
docker build  -t ${image} .
docker tag ${image} ${fullname}
# push to ecr
docker push ${fullname}

WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded
Sending build context to Docker daemon  103.9kB
Step 1/7 : FROM python:3.9
 ---> 7ef94ac333fa
Step 2/7 : RUN apt-get update
 ---> Using cache
 ---> 529128453704
Step 3/7 : RUN pip install --upgrade pip
 ---> Using cache
 ---> 8f56313abc92
Step 4/7 : COPY requirements.txt .
 ---> Using cache
 ---> e7a2b048faaa
Step 5/7 : RUN pip install -r requirements.txt
 ---> Using cache
 ---> b2a2198beeb6
Step 6/7 : COPY script.py .
 ---> edeee154e36f
Step 7/7 : ENTRYPOINT ["python3", "script.py"]
 ---> Running in b94202972b3c
Removing intermediate container b94202972b3c
 ---> 21e8cbc507d4
Successfully built 21e8cbc507d4
Successfully tagged genxii-ad-eval-valid:latest
The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-ad-eval-valid]
cb9db7114119: Preparing
996f52004b23: Preparing
d34f16843ccb: Preparing
7f7a9ee63288: Preparing
781f058a9424: Preparing
78ecb2a2f011: Preparing
84062ebc4cf5: Preparing
2180aea5f54b: Preparing
86388e04a96b: Preparing
893507f

### Clean-up

In [6]:
# rm files
for str_file in ['Dockerfile','requirements.txt','script.py']:
    try:
        os.remove(f'./{str_file}')
    except:
        pass